In [1]:
# 1. Implement a basic Statistical Machine Translation (SMT) model using word-by-word translation with a dictionary lookup approach

def load_translation_dictionary():
    return {
        'the': 'le',
        'cat': 'chat',
        'is': 'est',
        'on': 'sur',
        'mat': 'tapis'
    }

def translate_sentence(sentence, translation_dict):
    words = sentence.lower().split()
    translated_words = [translation_dict.get(word, word) for word in words]
    return ' '.join(translated_words)

# Example usage
translation_dict = load_translation_dictionary()
sentence = "The cat is on the mat"
translated_sentence = translate_sentence(sentence, translation_dict)
print(translated_sentence)  # Output: le chat est sur le tapis


le chat est sur le tapis


In [2]:
# 2. Implement an Attention mechanism in a Neural Machine Translation (NMT) model using PyTorch

import torch
import torch.nn as nn
import torch.optim as optim

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1)))
        return outputs, hidden, cell

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Parameter(torch.rand(hidden_dim))

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = torch.sum(self.v * energy, dim=2)
        return torch.nn.functional.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout, attention):
        super(Decoder, self).__init__()
        self.output_dim = output_dim
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.LSTM((hidden_dim * 2) + emb_dim, hidden_dim, n_layers, dropout=dropout)
        self.fc_out = nn.Linear((hidden_dim * 2) + hidden_dim + emb_dim, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.dropout(self.embedding(input))
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        weighted = torch.bmm(a, encoder_outputs.permute(1, 0, 2))
        rnn_input = torch.cat((embedded, weighted.permute(1, 0, 2)), dim=2)
        output, (hidden, cell) = self.rnn(rnn_input, (hidden, cell))
        prediction = self.fc_out(torch.cat((output.squeeze(0), weighted.squeeze(1), embedded.squeeze(0)), dim=1))
        return prediction, hidden, cell

# Example usage
INPUT_DIM = 1000
OUTPUT_DIM = 1000
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 512
N_LAYERS = 2
ENC_DROPOUT = 0.5
DEC_DROPOUT = 0.5

attn = Attention(HID_DIM)
enc = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM, N_LAYERS, ENC_DROPOUT)
dec = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM, N_LAYERS, DEC_DROPOUT, attn)


In [3]:
# 3. Use a pre-trained GPT model to perform machine translation from English to French

from transformers import pipeline

translator = pipeline("translation_en_to_fr", model="t5-small")
result = translator("The cat is on the mat")
print(result[0]['translation_text'])  # Output: Le chat est sur le tapis.


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


Le chat est sur le mat.


In [ ]:
# 4. Generate a short poem using GPT-2 for a specific theme (e.g., "Nature")

from transformers import pipeline

generator = pipeline("text-generation", model="gpt-2")
prompt = "In the heart of the forest"
poem = generator(prompt, max_length=50, num_return_sequences=1)
print(poem[0]['generated_text'])


In [ ]:
# 5. Implement a basic reinforcement learning setup for text generation using PyTorch's reward function

import torch
import torch.nn as nn
import torch.optim as optim

class SimpleRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        output = self.fc(output[:, -1, :])
        return output

# Example usage
VOCAB_SIZE = 1000
EMBED_SIZE = 256
HIDDEN_SIZE = 512
OUTPUT_SIZE = VOCAB_SIZE

model = SimpleRNN(VOCAB_SIZE, EMBED_SIZE, HIDDEN_SIZE, OUTPUT_SIZE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Dummy training loop
for epoch in range(100):
    optimizer.zero_grad()
    input_seq = torch.randint(0, VOCAB_SIZE, (1, 10))
    target_seq = torch.randint(0, VOCAB_SIZE, (1, 10))
    output = model(input_seq)
    loss = criterion(output, target_seq.view(-1))
    loss.backward()
    optimizer.step()


In [ ]:
# 6. Create a simple multimodal generative model that generates an image caption given an image

import torch
import torch.nn as nn
import torchvision.models as models
from transformers import BertTokenizer, BertModel

class ImageCaptioningModel(nn.Module):
    def __init__(self, encoder, decoder, embed_size, vocab_size, hidden_size):
        super(ImageCaptioningModel, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.fc = nn.Linear(encoder.fc.in_features, embed_size)
        self.embed = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, images, captions):
        features = self.encoder(images)
        features = self.fc(features).unsqueeze(1)
        embeddings = self.embed(captions)
        embeddings = torch.cat((features, embeddings), 1)
        lstm_out, _ = self.lstm(embeddings)
        outputs = self.fc_out(lstm_out)
        return outputs

# Example usage
encoder = models.resnet50(pretrained=True)
decoder = BertModel.from_pretrained("bert-base-uncased")
VOCAB_SIZE = 1000
EMBED_SIZE = 256
HIDDEN_SIZE = 512

model = ImageCaptioningModel(encoder, decoder, EMBED_SIZE, VOCAB_SIZE, HIDDEN_SIZE)


In [ ]:
# 7. Evaluate bias in generated content by analyzing GPT responses to prompts with potentially sensitive terms

from transformers import pipeline

generator = pipeline("text-generation", model="gpt-2")

prompts = ["Women are", "Men are", "Black people are", "White people are"]
responses = {}

for prompt in prompts:
    response = generator(prompt, max_length=50, num_return_sequences=1)
    responses[prompt] = response[0]['generated_text']

for prompt, response in responses.items():
    print(f"Prompt: {prompt}\nResponse: {response}\n")


In [6]:
# 8. Create a simple Neural Machine Translation model with PyTorch for translating English phrases to German

import torch
import torch.nn as nn
import torch.optim as optim

class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim, n_layers, dropout):
        super(Encoder, self).__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.LSTM(emb_dim, hidden_dim, n_layers, dropout=dropout)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.rnn(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim, n_layers, dropout):
        super(Decoder, self).__init__()
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.r